## RAG with Tables and Images
Source file: `RAG_Table_Image.pdf`. It has normal text, a native table (page 1),
a bar chart image and a pricing table that is only a screenshot (page 2).

| Content | How it is indexed | What a query returns |
|---|---|---|
| Text | chunked + embedded | the text chunk |
| Native table | one vector per row + one summary vector, all pointing to the same table | the **whole table** |
| Image | Claude vision describes it (chart values, tables inside the image), the description is embedded; the image file is saved locally | the **image itself** + its description |

At question time the matching text, tables and images are sent to Claude, which answers from them.

### Step 1: Imports and configuration

In [1]:
import os
import base64
import hashlib
import pdfplumber
from pypdf import PdfReader
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from IPython.display import display, Markdown, Image
import anthropic

load_dotenv()

PDF_PATH = "RAG_Table_Image.pdf"
IMAGE_FOLDER = "RAG_Table_Image_assets"   # extracted images are saved here
NAMESPACE = "rag-table-image"
os.makedirs(IMAGE_FOLDER, exist_ok=True)

c:\Users\DELL\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Connect to Pinecone

In [2]:
pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
index_name = os.getenv("Pinecone_Index_Name")

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)

### Step 3: Embedding model

In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def embed(texts):
    return model.encode(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1775.93it/s]


### Step 4: Claude client
Used to describe images (vision) and to answer questions.

In [ ]:
client = anthropic.Anthropic(api_key=os.getenv("Claude_API_KEY"))


def ask_claude(content):
    """Claude's text answer for a prompt (string or list of content blocks), or None if Claude declined."""
    response = client.beta.messages.create(
        model="claude-opus-5-5",
        max_tokens=16000,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",  # if Claude declines, the API retries on a fallback model
        messages=[{"role": "user", "content": content}],
    )
    if response.stop_reason == "refusal":
        return None
    return next(b.text for b in response.content if b.type == "text")


def image_block(path):
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")
    return {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": data}}

### Step 5: Extract native tables
Each table is kept as Markdown, with the line above it as its title.

In [ ]:
def to_markdown(rows):
    rows = [[(c or "").replace("\n", " ").strip() for c in r] for r in rows]
    header, *body = rows
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(r) + " |" for r in body]
    return "\n".join(lines)


def table_title(page, table, page_no):
    above = page.crop((0, max(0, table.bbox[1] - 30), page.width, table.bbox[1])).extract_text() or ""
    return above.strip().split("\n")[-1] if above.strip() else f"Table on page {page_no}"


tables = []   # dict per table: id, page, title, rows, markdown
with pdfplumber.open(PDF_PATH) as pdf:
    for page_no, page in enumerate(pdf.pages, start=1):
        for n, t in enumerate(page.find_tables(), start=1):
            rows = [[(c or "").replace("\n", " ").strip() for c in r] for r in t.extract()]
            tables.append({
                "id": f"table-p{page_no}-{n}",
                "page": page_no,
                "title": table_title(page, t, page_no),
                "rows": rows,
                "markdown": to_markdown(rows),
                "bbox": t.bbox,
            })

for t in tables:
    print(t["id"], "|", t["title"])
    display(Markdown(t["markdown"]))

### Step 6: Extract text (outside the tables, so table data isn't indexed twice) and chunk it

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", " ", ""])

text_chunks = []   # dict per chunk: id, page, text
with pdfplumber.open(PDF_PATH) as pdf:
    for page_no, page in enumerate(pdf.pages, start=1):
        for t in tables:
            if t["page"] == page_no:
                page = page.outside_bbox(t["bbox"])
        for i, chunk in enumerate(splitter.split_text(page.extract_text() or "")):
            text_chunks.append({"id": f"text-p{page_no}-{i}", "page": page_no, "text": chunk})

for c in text_chunks:
    print(f"\n--- {c['id']} ---\n{c['text']}")

### Step 7: Extract images
Save each embedded image as PNG; the file name is a hash of its bytes, so the same image always gets the same id.

In [ ]:
images = []   # dict per image: id, page, path
for page_no, page in enumerate(PdfReader(PDF_PATH).pages, start=1):
    for img in page.images:
        image_id = "img-" + hashlib.sha256(img.data).hexdigest()[:16]
        path = os.path.join(IMAGE_FOLDER, f"{image_id}.png")
        img.image.save(path, "PNG")
        images.append({"id": image_id, "page": page_no, "path": path})

for im in images:
    print(im["id"], "page", im["page"], im["path"])
    display(Image(filename=im["path"], width=500))

### Step 8: Describe images with Claude vision
The description is what gets embedded, so it must contain every number and label.
Images already in Pinecone reuse their stored description, so Claude is only called for new images.

In [ ]:
DESCRIBE_PROMPT = """Describe this image from a business document so it can be found by search.
- Start with the image's title.
- If it is a chart, list every label and value shown.
- If it contains a table, transcribe the full table as a Markdown table.
- Include any notes or callouts.
Only describe what is visible in the image."""


def stored_description(image_id):
    res = index.fetch(ids=[f"{image_id}#0"], namespace=NAMESPACE)
    v = res.vectors.get(f"{image_id}#0")
    return v.metadata.get("description") if v else None


for im in images:
    im["description"] = stored_description(im["id"])
    if im["description"]:
        print(f"{im['id']}: description already in Pinecone, skipping Claude")
    else:
        im["description"] = ask_claude([image_block(im["path"]), {"type": "text", "text": DESCRIBE_PROMPT}]) or ""
        print(f"{im['id']}: described by Claude")
    print(im["description"], "\n")

### Step 9: Build the records
Every record has `type` (text / table / image) and `item_id`. Several vectors can share one `item_id`
(the rows of a table, the chunks of an image description), so any of them leads back to the same table or image.

In [ ]:
records = []   # (id, text_to_embed, metadata)

for c in text_chunks:
    records.append((c["id"], c["text"], {
        "type": "text", "item_id": c["id"], "page": c["page"], "text": c["text"],
    }))

for t in tables:
    header, *body = t["rows"]
    table_meta = {"type": "table", "item_id": t["id"], "page": t["page"],
                  "title": t["title"], "table_markdown": t["markdown"]}
    # Summary vector: title + column names + first column values
    summary = f"{t['title']}. Columns: {', '.join(header)}. Rows: {', '.join(r[0] for r in body)}"
    records.append((f"{t['id']}#summary", summary, {**table_meta, "text": summary}))
    # One vector per row: "Column: value; Column: value ..."
    for r_no, row in enumerate(body, start=1):
        row_text = f"{t['title']}. " + "; ".join(f"{h}: {v}" for h, v in zip(header, row) if v)
        records.append((f"{t['id']}#row{r_no}", row_text, {**table_meta, "text": row_text}))

desc_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0, separators=["\n\n", "\n", " ", ""])
for im in images:
    for i, chunk in enumerate(desc_splitter.split_text(im["description"])):
        records.append((f"{im['id']}#{i}", chunk, {
            "type": "image", "item_id": im["id"], "page": im["page"],
            "image_path": im["path"], "description": im["description"], "text": chunk,
        }))

for rid, text, meta in records:
    print(f"{meta['type']:5} {rid}: {text[:80]}")

### Step 10: Embed and insert into Pinecone

In [ ]:
vectors = embed([text for _, text, _ in records])

batch = [
    {"id": rid, "values": vectors[i].tolist(), "metadata": meta}
    for i, (rid, _, meta) in enumerate(records)
]
for start in range(0, len(batch), 100):
    index.upsert(vectors=batch[start:start + 100], namespace=NAMESPACE)

print(f"Inserted {len(batch)} vectors: "
      f"{len(text_chunks)} text chunks, {len(tables)} tables, {len(images)} images")

### Step 11: Retrieve
Search, then group matches by `item_id` so a table or image is returned once, as a whole.

In [ ]:
def retrieve(question, top_k_items=3):
    res = index.query(
        vector=embed(question).tolist(),
        top_k=15,
        include_metadata=True,
        namespace=NAMESPACE,
    )
    items = {}
    for m in res.matches:   # matches come best first, so the first hit per item is its best score
        items.setdefault(m.metadata["item_id"], {**m.metadata, "score": m.score})
    return list(items.values())[:top_k_items]


def show(item):
    print(f"\n[{item['type'].upper()}] page {item['page']} | score {item['score']:.3f}")
    if item["type"] == "table":
        print(item["title"])
        display(Markdown(item["table_markdown"]))
    elif item["type"] == "image":
        display(Image(filename=item["image_path"], width=500))
    else:
        print(item["text"])

In [ ]:
for item in retrieve("Show me the subscription pricing table"):
    show(item)

### Step 12: Answer with Claude
The retrieved text and tables go in as text; retrieved images are sent as the images themselves, so Claude reads them directly.

In [ ]:
def answer(question):
    items = retrieve(question)

    content = []
    for n, it in enumerate(items, start=1):
        if it["type"] == "image":
            content.append({"type": "text", "text": f"[Source {n}] Image from page {it['page']}:"})
            content.append(image_block(it["image_path"]))
        elif it["type"] == "table":
            content.append({"type": "text", "text": f"[Source {n}] Table from page {it['page']}: {it['title']}\n{it['table_markdown']}"})
        else:
            content.append({"type": "text", "text": f"[Source {n}] Text from page {it['page']}:\n{it['text']}"})

    content.append({"type": "text", "text": f"""Answer the question using only the sources above.
Cite the source numbers you used, e.g. [Source 2]. Do not change any numbers.
If the sources don't contain the answer, say so.

Question: {question}"""})

    reply = ask_claude(content) or "Sorry, I can't answer that question."

    print("Question:", question)
    print("\nAnswer:\n" + reply)
    print("\nSources:")
    for it in items:
        show(it)

Question about the native table (page 1)

In [ ]:
answer("Who heads the Finance department and what is its budget?")

Question about the chart image (page 2)

In [ ]:
answer("Which region had the highest revenue in FY2025?")

Question about the table inside an image (page 2)

In [ ]:
answer("How much is the Business plan per month and what is its support SLA?")